# 🧱 第2步：手搭 CNN 卷积神经网络

| 序号 | 知识点 | 要搞懂什么 |
|------|--------|-----------|
| ① | **Conv2d** | 卷积核、通道、padding、stride |
| ② | **MaxPool2d** | 怎么缩小特征图？ |
| ③ | **BatchNorm + Dropout** | 怎么防过拟合、加速训练？ |
| ④ | **ReLU** | 为什么要加非线性？ |
| ⑤ | **尺寸公式** | 每层 H×W 怎么算？ |
| ⑥ | **参数量** | 模型多大？显存够不够？ |

> 你会亲手搭一个处理 100 类的 CNN，约 3M 参数，M5 完全跑得动。

---
## Cell 1：导入库 & 设置设备

In [1]:
import torch
import torch.nn as nn

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 使用 Apple MPS (GPU) 加速")
else:
    device = torch.device("cpu")
    print("🐢 使用 CPU")

🚀 使用 Apple MPS (GPU) 加速


---
## Cell 2：理解卷积层 Conv2d

### 卷积在做什么？

拿一个 3×3 的卷积核在图片上滑动。每到一处，把卷积核权重和图片像素做"点积"，得到一个值。滑完整张图 = 一张"特征图"。64 个卷积核 = 64 张特征图。

```
输入 32×32×3  →  Conv2d(3→64, kernel=3, padding=1)  →  输出 32×32×64
```

### 关键参数

| 参数 | 含义 | 这里用 |
|------|------|--------|
| in_channels | 输入通道数 | 3 (RGB) |
| out_channels | 输出通道数（卷积核个数） | 64 |
| kernel_size | 卷积核大小 | 3 |
| padding | 边缘补 0 圈数 | 1 |
| stride | 滑动步长 | 1 |

### 🔑 输出尺寸公式

```
H_out = (H_in - kernel + 2×padding) / stride + 1
      = (32 - 3 + 2) / 1 + 1 = 32   ← padding=1 刚好保持尺寸不变！
```

In [2]:
# 创建一个 Conv2d 层测试
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

x = torch.randn(1, 3, 32, 32)   # 假图片
out = conv(x)
print(f"输入:  {x.shape}")
print(f"输出:  {out.shape}")
print(f"  → 尺寸不变(32→32)，通道变了(3→16)")

# 参数量
n = sum(p.numel() for p in conv.parameters())
print(f"参数量: {n}  = 16个核 × (3×3×3权重 + 1偏置)")

输入:  torch.Size([1, 3, 32, 32])
输出:  torch.Size([1, 16, 32, 32])
  → 尺寸不变(32→32)，通道变了(3→16)
参数量: 448  = 16个核 × (3×3×3权重 + 1偏置)


---
## Cell 3：理解池化 MaxPool2d

池化 = 缩小特征图。在 2×2 窗口里取最大值，stride=2。

```
4×4 → MaxPool(2×2, stride=2) → 2×2   尺寸减半！
```

**作用：** 减少计算量 + 增大感受野 + 轻微平移不变性

In [3]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)
x = torch.randn(1, 16, 32, 32)
out = pool(x)
print(f"池化前: {x.shape}")
print(f"池化后: {out.shape}")
print(f"  → 高宽减半，通道不变")

池化前: torch.Size([1, 16, 32, 32])
池化后: torch.Size([1, 16, 16, 16])
  → 高宽减半，通道不变


---
## Cell 4：理解 BatchNorm 和 Dropout

### BatchNorm
把每层输出重新拉到均值=0、标准差=1。**稳定训练，允许更大学习率。**

### Dropout
训练时随机丢弃一部分神经元（比如 30%）。**防止网络过度依赖少数神经元（过拟合）。**

> Dropout 只在训练时生效，测试时自动关闭。

---
## Cell 5：搭积木 —— 定义 ConvBlock

CNN 通用模式：**两次卷积 + BN + ReLU + 池化 + Dropout = 一个 Block**

In [5]:
class ConvBlock(nn.Module): #python语法：class 类(父类)这样类就可以继承父类的所有属性和方法
    """一个标准卷积模块：双卷积 + BN + ReLU + 池化 + Dropout"""
    def __init__(self, in_ch, out_ch, dropout=0.3): #输入通道数，输出通道数（即卷积核个数），dropout比率，为什么没有输入和kernel大小？
        super().__init__() #相当于去找父类nn.module的初始化方法
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),   # 尺寸减半
            nn.Dropout(dropout)    # 防过拟合
        ) #注意这里再init函数用.block就创建了神经网络结构，只与kernel尺寸、in_ch、out_ch有关，而输入H W是数据的范畴
        #一共有in_ch*out_ch个二维kernel，有in_ch*out_ch*9个权重参数（不考虑偏置的话）
    def forward(self, x): return self.block(x) #此为pytorch强制规定的实现计算的逻辑，用forward来规定数据流移动，在外部直接用b()即可

# 测试
b = ConvBlock(3, 64)
x = torch.randn(1, 3, 32, 32)
out = b(x) #可以直接将类当函数使用，实际数据逻辑在forward函数中实现
print(f"输入: {x.shape}  →  输出: {out.shape}")
print(f"通道: 3→64, 尺寸: 32→16 (减半!)")

输入: torch.Size([1, 3, 32, 32])  →  输出: torch.Size([1, 64, 16, 16])
通道: 3→64, 尺寸: 32→16 (减半!)


---
## Cell 6：组装完整 CNN

```
输入 [B, 3, 32, 32]
  ├─ Block(3→64)     → [B, 64, 16, 16]
  ├─ Block(64→128)   → [B, 128, 8, 8]
  ├─ Block(128→256)  → [B, 256, 4, 4]
  ├─ Block(256→512)  → [B, 512, 2, 2]
  ├─ Flatten         → [B, 2048]
  ├─ Linear(2048→256) + ReLU + Dropout(0.5)
  ├─ Linear(256→100) → [B, 100]  ← 100个类得分
```

🧠 黄金法则：**通道翻倍，尺寸减半**——信息从空间细节流向语义理解。

In [7]:
class CIFAR100_CNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential( #feature和classifier一起出现
            ConvBlock(3, 64,  dropout=0.2),
            ConvBlock(64, 128, dropout=0.3),
            ConvBlock(128, 256, dropout=0.4),
            ConvBlock(256, 512, dropout=0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), #先将数据变为一维向量
            nn.Linear(512*2*2, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), #nn.Linear全连接层
            nn.Linear(256, num_classes),
        ) #这里计算交叉熵损失函数时才会自动将softmax层加上
    def forward(self, x):
        return self.classifier(self.features(x))

model = CIFAR100_CNN().to(device)
x = torch.randn(1, 3, 32, 32).to(device)
out = model(x)
print(f"输入: {x.shape}  →  输出: {out.shape}")
print(f"输出是 100 个得分，最高分 = 预测类别")

输入: torch.Size([1, 3, 32, 32])  →  输出: torch.Size([1, 100])
输出是 100 个得分，最高分 = 预测类别


---
## Cell 7：参数量统计

CIFAR-100 有 50000 训练样本，约 3M 参数是个合适的起点。

In [9]:
total = sum(p.numel() for p in model.parameters())
mem = total * 4 / 1024**2  # float32
print(f"总参数: {total:,}")
print(f"权重占用: {mem:.1f} MB")
print(f"训练时 ~{mem*3:.0f}MB (含梯度+优化器)")
print("16GB 内存 ✅")

总参数: 5,239,460
权重占用: 20.0 MB
训练时 ~60MB (含梯度+优化器)
16GB 内存 ✅


---
## Cell 8：追踪数据流 —— 看每层 shape 变化

In [10]:
print(f"{'层':35s} {'输出 shape'}")
print("-"*55)
x = torch.randn(4, 3, 32, 32).to(device)
print(f"{'输入':35s} {list(x.shape)}")
for i, block in enumerate(model.features):
    x = block(x)
    print(f"Block{i+1} (ch={x.shape[1]}){'':20s} {list(x.shape)}")
for layer in model.classifier:
    x = layer(x)
    n = type(layer).__name__
    if n == 'Linear': n = f'Linear({layer.in_features}→{layer.out_features})'
    print(f"{n:<35s} {list(x.shape)}")
print("\n✅ 全对！")

层                                   输出 shape
-------------------------------------------------------
输入                                  [4, 3, 32, 32]
Block1 (ch=64)                     [4, 64, 16, 16]
Block2 (ch=128)                     [4, 128, 8, 8]
Block3 (ch=256)                     [4, 256, 4, 4]
Block4 (ch=512)                     [4, 512, 2, 2]
Flatten                             [4, 2048]
Linear(2048→256)                    [4, 256]
ReLU                                [4, 256]
Dropout                             [4, 256]
Linear(256→100)                     [4, 100]

✅ 全对！


---
## 🎉 第2步完成！

| ✅ | 知识点 | 一句话 |
|----|--------|--------|
| ① | **Conv2d** | 卷积核滑窗提取特征 |
| ② | **MaxPool2d** | 2×2取最大，尺寸减半 |
| ③ | **BatchNorm** | 稳定输出分布，加速训练 |
| ④ | **Dropout** | 随机丢神经元，防过拟合 |
| ⑤ | **ReLU** | max(0,x)，引入非线性 |
| ⑥ | **尺寸公式** | (H-K+2P)/S+1 |
| ⑦ | **黄金法则** | 通道↑尺寸↓，空间→语义 |

> 📝 下一步：训练循环——Loss、反向传播、优化器，模型开始真正"学习"！